# Lekcija 13 - Spomin agenta


## Namestitev

Ta zvezek prikazuje, kako ustvariti agenta za rezervacijo potovanj z **trajnim spominom** z uporabo **Microsoft Agent Framework** (MAF).

Naučili se boste, kako različne vrste spomina agenta — delovni, kratkoročni in dolgoročni — vplivajo na to, kako agent ohranja in uporablja informacije skozi pogovore.

**Zahteve:**
- Projekt Azure AI Foundry z nameščenim klepetalnim modelom (npr. `gpt-4o-mini`).
- Prijava z Azure CLI — zaženite `az login` v terminalu.
- `AZURE_AI_PROJECT_ENDPOINT` — končna točka vašega projekta Azure AI Foundry.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ime vašega nameščenega modela.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Vrste spomina agenta

AI agenti lahko uporabljajo različne vrste spomina, od katerih ima vsaka poseben namen:

### Delovni spomin
Samo nit pogovora — sporočila, izmenjana v eni seji. Agent se lahko sklicuje na prejšnja sporočila v isti niti, da ohrani koherenco. V MAF je to ustvarjeno preko **`agent.create_session()`**, ki vrne `AgentSession`.

### Kratkoročni spomin
Informacije, ki trajajo skozi trajanje naloge ali seje, vendar niso shranjene trajno. Na primer, agent lahko med večkrožnim načrtovalnim pogovorom zbira dejstva in jih uporabi za izdelavo končnega itinerarja.

### Dolgoročni spomin
Preference in dejstva, ki vztrajajo **prek sej**. Uporabnik, ki se vrne, ne bi smel ponavljati svojih prehranskih omejitev ali načina potovanja. Dolgoročni spomin je običajno podprt z zunanjim shranjevalnikom — podatkovno bazo, datoteko ali vektorskim indeksom — in je agentu dostopen preko orodij.


## Delovni pomnilnik s sejami

Najpreprostejša oblika pomnilnika je pogovorna seja. Ko isto sejo (ustvarjeno z `agent.create_session()`) posredujete zaporednim klicem `agent.run()`, agent vidi celotno zgodovino tega pogovora in se lahko spomni prej omenjenih podrobnosti.

Ustvarimo potovalnega agenta in pokažimo delovni pomnilnik.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Agent je pravilno priklical proračun, ker imata obe sporočili isto sejo. To je **delovni pomnilnik** — obstaja le za čas trajanja seje.

### Kaj se zgodi z novo nitjo?

Če ustvarimo **novo** sejo, agent nima spomina na prejšnji pogovor:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Vzorec dolgoročnega spomina

Za pomnjenje uporabniških nastavitev **čez seje** potrebujemo trajno shrambo, ki živi zunaj pogovorne zanke. Agent dostopa do te shrambe prek **orodij** — funkcij, ki jih lahko kliče za shranjevanje in pridobivanje informacij.

Spodaj implementiramo enostavno shrambo nastavitev v pomnilniku (v produkciji bi za to uporabili podatkovno bazo ali vektorski indeks) in jo predstavimo kot orodja, ki jih agent lahko uporablja.

### Arhitektura
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenarij 1 — Uporabnik, ki prvič rezervira potovanje za obletnico

Sarah obišče prvič. Agent naj shrani njene preference preko orodij in jih uporabi za priporočanje hotelov.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scenarij 2 — Sarah se vrne čez tedne

Sarah začne **popolnoma novo nit** (simulira novo sejo). Delovni pomnilnik je prazen, vendar ima dolgoročni shrambnik nastavitev še vedno njene podatke. Agent jih mora pridobiti in uporabiti za prilagajanje priporočil.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Povzetek

V tej lekciji ste spoznali tri vrste pomnilnika agenta in kako jih implementirati z Microsoft Agent Framework:

| Vrsta pomnilnika | Mehanizem MAF | Življenjska doba |
|---|---|---|
| **Delovni** | `agent.create_session()` | En pogovor |
| **Kratkoročni** | Akumuliran kontekst znotraj niti | Ena naloga / seja |
| **Dolgoročni** | Zunanji skladiščni dostop prek funkcij `@tool` | Čez seje |

### Ključne spoznanja
1. **`agent.create_session()`** zagotavlja delovni pomnilnik — agent vidi celotno zgodovino pogovora znotraj seje.
2. **Nove seje izgubijo kontekst** — brez dolgoročnega pomnilnika agent ne more priklicati prejšnjih pogovorov.
3. **Funkcije `@tool`** premoščajo vrzel — omogočajo agentu shranjevanje in pridobivanje informacij iz trajnega skladišča.
4. **Personalizacija se izboljšuje s časom** — več ko je shranjenih preferenc, bolje so priporočila agenta.

### Praktične uporabe
- **Pomoč strankam**: Shrani zgodovino in preference stranke
- **Osebni asistenti**: Ohranja kontekst čez dneve ali tedne
- **Zdravstvo**: Sledi informacijam in preferencam pacientov
- **E-trgovina**: Personalizirano nakupovanje glede na zgodovino

### Naslednji koraki
- Zamenjajte slovar v pomnilniku z bazo podatkov ali vektorskim skladiščem (npr. Azure AI Search)
- Dodajte potek veljavnosti pomnilnika za časovno občutljive informacije
- Zgradite sisteme z več agenti s skupnim pomnilnikom
- Raziskujte Cognee zvezek za pomnilnik, podprt z znanstvenim grafom


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Opozorilo**:
Ta dokument je bil preveden z uporabo storitve za prevajanje z umetno inteligenco [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, upoštevajte, da lahko avtomatizirani prevodi vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvorni jezik je treba šteti za avtoritativni vir. Za ključne informacije priporočamo strokovni človeški prevod. Ne prevzemamo odgovornosti za kakršne koli nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
